In [1]:
import pandas as pd
import kagglehub
import ast
import numpy as np
from collections import Counter

In [2]:
path_ml = kagglehub.dataset_download("grouplens/movielens-latest-full")
path_tmdb = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("MovieLens path:", path_ml)
print("TMDB path:", path_tmdb)

# MovieLens files
ratings_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv')
movies_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv')
links_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv')
tags_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv')

# TMDB files
metadata_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv', low_memory=False)
credits_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv')
keywords_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv')

MovieLens path: /Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1
TMDB path: /Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7


In [3]:
def get_director(crew):
    return next((p['name'] for p in crew if p.get('job') == 'Director'), None)

def get_producer(crew):
    return next((p['name'] for p in crew if p.get('job') == 'Producer'), None)

def get_lead_actor(cast):
    return next((p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf'))) if 'name' in p), None)

def get_gender_of_lead(cast, lead_name):
    return next((p['gender'] for p in cast if p.get('name') == lead_name), None)

def get_other_lead(cast, lead_name):
    lead_gender = get_gender_of_lead(cast, lead_name)
    opposite_gender = {1: 2, 2: 1}.get(lead_gender)
    return next(
        (p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf')))
         if p.get('name') != lead_name and p.get('gender') == opposite_gender),
        None
    )

def get_other_actors(cast, exclude_names, max_count=3):
    return [
        p['name'] for p in sorted(cast, key=lambda x: x.get('order', float('inf')))
        if p.get('name') not in exclude_names and p.get('name') is not None
    ][:max_count]


In [4]:
# Clean links
links_ml = links_ml[links_ml['tmdbId'].notnull()]
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

# Clean metadata
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)
metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)

# Parse credits
credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)

credits_tmdb['producer'] = credits_tmdb['crew'].apply(get_producer)
credits_tmdb['tmdbId'] = credits_tmdb['id']
credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['producer'] = credits_tmdb['crew'].apply(get_producer)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)
credits_tmdb['other_lead'] = credits_tmdb.apply(
    lambda row: get_other_lead(row['cast'], row['lead_actor']), axis=1
)
credits_tmdb['other_actors'] = credits_tmdb.apply(
    lambda row: get_other_actors(row['cast'], exclude_names={row['lead_actor'], row['other_lead']}), axis=1
)

# Parse keywords
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)])
keywords_tmdb['tmdbId'] = keywords_tmdb['id']

# Aggregate tags
tags_agg = tags_ml.groupby('movieId')['tag'].apply(lambda x: list(set(x))).reset_index()

# Ratings stats
rating_stats = ratings_ml.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']

In [5]:
movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})
movies_full = pd.merge(movies_ml_links, metadata_tmdb, on='tmdbId', how='inner')

movies_full = pd.merge(movies_full, credits_tmdb[['tmdbId', 'director', 'lead_actor']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, keywords_tmdb[['tmdbId', 'keywords']], on='tmdbId', how='left')
movies_full = pd.merge(movies_full, tags_agg, on='movieId', how='left')
movies_full = pd.merge(movies_full, rating_stats, on='movieId', how='left')

In [6]:
# Runtime binning
bin_edges = list(range(0, 301, 30)) + [np.inf]
labels = [f'{bin_edges[i]}–{bin_edges[i+1]}min' if bin_edges[i+1] != np.inf else f'{bin_edges[i]}min+' for i in range(len(bin_edges) - 1)]
movies_full['runtime_bin'] = pd.cut(movies_full['runtime'], bins=bin_edges, labels=labels)

# Release dates
movies_full['release_date_parsed'] = pd.to_datetime(movies_full['release_date'], errors='coerce')
release_year_tmdb = movies_full['release_date_parsed'].dt.year
release_year_ml = movies_full['title_x'].str.extract(r'\((\d{4})\)')[0].astype(float)

movies_full['release_year_tmdb'] = release_year_tmdb
movies_full['release_year_ml'] = release_year_ml
movies_full['release_year'] = release_year_tmdb.combine_first(release_year_ml)
movies_full['release_year_merged'] = movies_full[['release_year_tmdb', 'release_year_ml']].min(axis=1)

# Ratings
movies_full['vote_count'] = movies_full[['vote_count_x', 'vote_count_y']].max(axis=1)
movies_full['vote_average'] = movies_full.apply(
    lambda row: row['vote_average_x'] if row['vote_count_x'] >= row['vote_count_y'] else row['vote_average_y'],
    axis=1
)

# Title
movies_full['title'] = movies_full['title_y'].combine_first(movies_full['title_x'])

# Genres
movies_full['genres_x_list'] = movies_full['genres_x'].fillna('').apply(lambda x: x.split('|') if isinstance(x, str) else [])
movies_full['genres_y_list'] = movies_full['genres_y'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)] if isinstance(x, list) else [])
movies_full['genre_list'] = movies_full.apply(lambda row: sorted(set(row['genres_x_list']) | set(row['genres_y_list'])), axis=1)

# Main genre (prefer TMDB, fallback to MovieLens)
def extract_main_genre(genres_y): return genres_y[0]['name'] if isinstance(genres_y, list) and len(genres_y) > 0 and isinstance(genres_y[0], dict) else None
movies_full['main_genre'] = movies_full['genres_y'].apply(extract_main_genre)
movies_full['main_genre'] = movies_full.apply(lambda row: row['main_genre'] if pd.notnull(row['main_genre']) else (row['genres_x_list'][0] if row['genres_x_list'] else None), axis=1)

# Additional features
movies_full['release_month'] = movies_full['release_date_parsed'].dt.month
movies_full['release_decade'] = (movies_full['release_year'] // 10) * 10
movies_full['popularity_score'] = movies_full['vote_average'] * np.log1p(movies_full['vote_count'])

In [7]:
all_genres = movies_full['genre_list'].explode()
genre_counts = Counter(all_genres)
genre_count_df = pd.DataFrame(genre_counts.items(), columns=['genre', 'count']).sort_values(by='count', ascending=False)
genre_count_df.head(10)  # Top 10 genres

,genre,count
7,Drama,23095
3,Comedy,14632
10,Thriller,8965
6,Romance,8238
8,Action,7554
9,Crime,5399
11,Horror,5073
17,Documentary,4415
0,Adventure,4412
13,Mystery,3205


In [ ]:
movies_full = movies_full.drop(columns=[
    'title_x', 'title_y',
    'vote_average_x', 'vote_average_y',
    'vote_count_x', 'vote_count_y',
    'release_year_tmdb', 'release_year_ml',
    'release_date',
    'genres_x', 'genres_y'
])

movies_full = movies_full.rename(columns={'release_date_parsed': 'release_date'})

In [9]:
pd.set_option('display.max_columns', None)

In [10]:
movies_full["production_countries"][3]

"[{'iso_3166_1': 'US', 'name': 'United States of America'}]"

In [11]:
movies_full["production_countries"].unique()

array(["[{'iso_3166_1': 'US', 'name': 'United States of America'}]",
       "[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]",
       "[{'iso_3166_1': 'GB', 'name': 'United Kingdom'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]",
       ...,
       "[{'iso_3166_1': 'PL', 'name': 'Poland'}, {'iso_3166_1': 'CZ', 'name': 'Czech Republic'}, {'iso_3166_1': 'SK', 'name': 'Slovakia'}]",
       "[{'iso_3166_1': 'CU', 'name': 'Cuba'}, {'iso_3166_1': 'DE', 'name': 'Germany'}, {'iso_3166_1': 'ES', 'name': 'Spain'}]",
       "[{'iso_3166_1': 'EG', 'name': 'Egypt'}, {'iso_3166_1': 'IT', 'name': 'Italy'}, {'iso_3166_1': 'US', 'name': 'United States of America'}]"],
      shape=(2391,), dtype=object)

In [12]:
movies_full.columns

Index(['movieId', 'imdbId', 'tmdbId', 'adult', 'belongs_to_collection',
       'budget', 'homepage', 'imdb_id', 'original_language', 'original_title',
       'overview', 'popularity', 'poster_path', 'production_companies',
       'production_countries', 'revenue', 'runtime', 'spoken_languages',
       'status', 'tagline', 'video', 'director', 'lead_actor', 'keywords',
       'tag', 'vote_min', 'vote_max', 'runtime_bin', 'release_date',
       'release_year', 'release_year_merged', 'vote_count', 'vote_average',
       'title', 'genres_x_list', 'genres_y_list', 'genre_list', 'main_genre',
       'release_month', 'release_decade', 'popularity_score'],
      dtype='object')

In [13]:
movies_full["spoken_languages"].unique()

array(["[{'iso_639_1': 'en', 'name': 'English'}]",
       "[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",
       "[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'es', 'name': 'Español'}]",
       ...,
       "[{'iso_639_1': 'sv', 'name': 'svenska'}, {'iso_639_1': 'de', 'name': 'Deutsch'}]",
       "[{'iso_639_1': 'ar', 'name': 'العربية'}, {'iso_639_1': 'pl', 'name': 'Polski'}]",
       "[{'iso_639_1': 'ff', 'name': 'Fulfulde'}, {'iso_639_1': 'en', 'name': 'English'}]"],
      shape=(1932,), dtype=object)